# 06. Segmentation and labeled objects

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner  
**Estimated study time:** 30–60 minutes

## What you will learn
- Convert masks to labeled objects
- Overlay labels on the image
- Use watershed as an optional tool for touching objects

> **Course habit:** run one cell at a time, inspect the result, and change one parameter before moving on.

## From a mask to individual objects

A Boolean mask says foreground/background. A **label image** assigns a unique integer to each connected object.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage as ski

def show_image(image, title='', cmap=None, figsize=(6, 5)):
    """Display one image with a clean, repeatable layout."""
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
from skimage import filters, measure, morphology, segmentation, color
image = ski.data.coins()
smooth = filters.gaussian(image, sigma=1.0)
threshold = filters.threshold_otsu(smooth)
mask = smooth > threshold
mask = morphology.remove_small_objects(mask, max_size=99)
labels = measure.label(mask)
print('foreground labels:', labels.max())

## Visualize labels

In [ ]:
overlay = color.label2rgb(labels, image=image, bg_label=0, alpha=0.35)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,img,title in zip(axes,[image,mask,overlay],['Original','Mask','Labeled overlay']):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## What if objects touch?

Connected-component labeling merges touching foreground pixels. Watershed is one common next step, but use it only when touching objects are a real problem.

In [ ]:
from scipy import ndimage as ndi
from skimage.feature import peak_local_max

distance = ndi.distance_transform_edt(mask)
coords = peak_local_max(distance, min_distance=15, labels=mask)
markers = np.zeros_like(mask, dtype=int)
markers[tuple(coords.T)] = np.arange(1, len(coords)+1)
separated = segmentation.watershed(-distance, markers, mask=mask)
print('objects after watershed:', separated.max())

## Exercise
Change `min_distance` and observe how the number of separated regions changes.

## Takeaway
Segmentation should match the biological or physical definition of the object you intend to measure.